# **1. Tiền xử lý dữ liệu**

Đọc hai file `store.csv` và `train.csv`, ghép lại, xử lý giá trị thiếu, loại ngày đóng cửa và lưu dữ liệu sạch cho các notebook sau.

In [1]:
# [UPDATED] Gom import, đường dẫn và hằng số lên đầu; bỏ các lệnh %pip install (cài thư viện bằng requirements.txt)
from pathlib import Path

import pandas as pd

PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
NO_COMPETITOR_DISTANCE = 999999  # [UPDATED] Giá trị điền cho cửa hàng không có thông tin đối thủ

**1.1 Đọc dữ liệu cửa hàng**

In [2]:
store_data = pd.read_csv('store.csv')  # [UPDATED] Đổi tên info_data -> store_data cho rõ nghĩa
print(store_data.shape)
store_data.head()

(1115, 10)


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [3]:
store_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment                 1115 non-null   str    
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    str    
dtypes: float64(5), int64(2), str(3)
memory usage: 87.2 KB


**1.2 Đọc dữ liệu doanh thu theo ngày**

In [4]:
# [UPDATED] Đọc StateHoliday dạng chữ (tránh lẫn 0 và '0') và đọc Date dạng ngày ngay từ đầu
train_data = pd.read_csv('train.csv', dtype={'StateHoliday': str}, parse_dates=['Date'])
print(train_data.shape)
train_data.head()

(1017209, 9)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [5]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   Store          1017209 non-null  int64         
 1   DayOfWeek      1017209 non-null  int64         
 2   Date           1017209 non-null  datetime64[us]
 3   Sales          1017209 non-null  int64         
 4   Customers      1017209 non-null  int64         
 5   Open           1017209 non-null  int64         
 6   Promo          1017209 non-null  int64         
 7   StateHoliday   1017209 non-null  str           
 8   SchoolHoliday  1017209 non-null  int64         
dtypes: datetime64[us](1), int64(7), str(1)
memory usage: 69.8 MB


In [6]:
# [UPDATED] Đếm số ngày có dữ liệu của mỗi cửa hàng để phát hiện cửa hàng bị thiếu dữ liệu
days_per_store = train_data.groupby('Store')['Date'].nunique()
print(days_per_store.value_counts())
missing_period = pd.date_range(train_data['Date'].min(), train_data['Date'].max()).difference(
    train_data.loc[train_data['Store'] == days_per_store.idxmin(), 'Date'])
print('Khoảng ngày bị thiếu:', missing_period.min().date(), '->', missing_period.max().date())

Date
942    934
758    180
941      1
Name: count, dtype: int64
Khoảng ngày bị thiếu: 2014-07-01 -> 2014-12-31


**1.3 Ghép hai bảng theo mã cửa hàng**

In [7]:
my_data = pd.merge(store_data, train_data, on='Store', how='left')
my_data = my_data.sort_values(by=['Store', 'Date']).reset_index(drop=True)
print(my_data.shape)

(1017209, 18)


In [8]:
my_data.describe().T

,count,mean,min,25%,50%,75%,max,std
Store,1017209.0,558.429727,1.0,280.0,558.0,838.0,1115.0,321.908651
CompetitionDistance,1014567.0,5430.085652,20.0,710.0,2330.0,6890.0,75860.0,7715.3237
CompetitionOpenSinceMonth,693861.0,7.222866,1.0,4.0,8.0,10.0,12.0,3.211832
CompetitionOpenSinceYear,693861.0,2008.690228,1900.0,2006.0,2010.0,2013.0,2015.0,5.992644
Promo2,1017209.0,0.500564,0.0,0.0,1.0,1.0,1.0,0.5
Promo2SinceWeek,509178.0,23.269093,1.0,13.0,22.0,37.0,50.0,14.095973
Promo2SinceYear,509178.0,2011.752774,2009.0,2011.0,2012.0,2013.0,2015.0,1.66287
DayOfWeek,1017209.0,3.998341,1.0,2.0,4.0,6.0,7.0,1.997391
Date,1017209,2014-04-11 01:30:42.846062,2013-01-01 00:00:00,2013-08-17 00:00:00,2014-04-02 00:00:00,2014-12-12 00:00:00,2015-07-31 00:00:00,NaN
Sales,1017209.0,5773.818972,0.0,3727.0,5744.0,7856.0,41551.0,3849.926175


**1.4 Xử lý giá trị thiếu**

In [9]:
my_data.isnull().sum()

Store                             0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
dtype: int64

Các ô trống đều có lý do: cửa hàng không tham gia Promo2 thì không có ngày bắt đầu Promo2; một số cửa hàng không rõ thông tin đối thủ. Vì vậy điền giá trị mặc định thay vì xoá dòng.

In [10]:
my_data['CompetitionDistance'] = my_data['CompetitionDistance'].fillna(NO_COMPETITOR_DISTANCE)
fill_zero_cols = ['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2SinceWeek', 'Promo2SinceYear']
my_data[fill_zero_cols] = my_data[fill_zero_cols].fillna(0)
my_data['PromoInterval'] = my_data['PromoInterval'].fillna('missing')
print('Số ô còn thiếu:', my_data.isnull().sum().sum())

Số ô còn thiếu: 0


**1.5 Kiểm tra trùng lặp**

In [11]:
print('Số dòng trùng lặp hoàn toàn:', my_data.duplicated().sum())
# [UPDATED] In ra số lượng thay vì hiển thị một bảng rỗng
print('Số cặp (Store, Date) bị lặp:', my_data.duplicated(subset=['Store', 'Date']).sum())

Số dòng trùng lặp hoàn toàn: 0
Số cặp (Store, Date) bị lặp: 0


**1.6 Loại ngày đóng cửa**

Ngày đóng cửa có doanh thu bằng 0 và không cần dự đoán. Sau khi loại, cột `Open` luôn bằng 1 nên cũng bỏ đi.

In [12]:
print('Số dòng đóng cửa (Open = 0):', (my_data['Open'] == 0).sum())
print('Số dòng mở cửa nhưng doanh thu = 0:', ((my_data['Open'] == 1) & (my_data['Sales'] == 0)).sum())
my_data = my_data.drop(columns='Open')
my_data = my_data[my_data['Sales'] > 0].reset_index(drop=True)
print('Kích thước sau khi lọc:', my_data.shape)

Số dòng đóng cửa (Open = 0): 172817
Số dòng mở cửa nhưng doanh thu = 0: 54
Kích thước sau khi lọc: (844338, 17)


**1.7 Lưu dữ liệu sạch**

In [13]:
# [UPDATED] Lưu dữ liệu sạch để các notebook sau đọc lại (trước đây thiếu bước này nên notebook 02 không chạy được)
my_data.to_csv(PROCESSED_DIR / 'rossmann_clean.csv', index=False)
print('Đã lưu', PROCESSED_DIR / 'rossmann_clean.csv')

Đã lưu data/processed/rossmann_clean.csv
